In [3]:
import pandas as pd
import numpy as np

#carregamento de dados brutos e bagunçados
df = pd.read_csv(r"D:\projetos git\analise-de-times-br\data\raw\campeonato-brasileiro-full.csv")


#tratamento de valores nulos
fake_null_values = ["", " ", "N/A", "-", "NULL", "null", "NaN", "nan"]
df = df.replace(fake_null_values, np.nan)

#limpar espaços em branco das colunas
colunas_texto = df.select_dtypes(include="object").columns
for coluna in colunas_texto:
    df[coluna] = df[coluna].str.strip()

#padronizar maisculas e minusculas
coluna_de_sigla = ["mandante_Estado", "visitante_Estado"]
coluna_de_nomes = ["mandante", "visitante", "arena", "tecnico_mandante", "tecnico_visitante"]    
for coluna in coluna_de_nomes:
    df[coluna] = df[coluna].str.title()

for coluna in coluna_de_sigla:
    df[coluna] = df[coluna].str.upper() 


#correção da  coluna vencedor
df["vencedor"] = df["vencedor"].fillna("Empate")
df["vencedor"] = df["vencedor"].str.title()

#correção da coluna de data
formatos_de_data =  ["%d/%m/%Y", "%Y-%m-%d", "%d.%m.%Y", "%d-%m-%Y","%m/%d/%Y", "%d/%m/%y", "%Y/%m/%d"]
data_convertida = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

for formato in formatos_de_data:
    ainda_sem_data = data_convertida.isna()
    segunda_tentativa = pd.to_datetime(df.loc[ainda_sem_data, "data"], format = formato, errors="coerce")
    data_convertida.loc[ainda_sem_data] = segunda_tentativa
df["data"] = data_convertida
datas_invalidas = df["data"].isna().sum()

display(df)

#correção as colunas de placar 
for coluna in ["mandante_Placar", "visitante_Placar"]:
    df[coluna] = df[coluna].str.replace(" un","", regex=False)
    df[coluna] = df[coluna].str.replace(",",".", regex=False)
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

#trata outliers valores absurdos
limite_maximo_de_gols = 12
for coluna in ["mandante_Placar", "visitante_Placar"]:
    outliers = df[coluna] > limite_maximo_de_gols
    df.loc[outliers, coluna] = np.nan 

duplicatas = df.duplicated(subset="ID").sum()
df = df.drop_duplicates(subset = "ID", keep = "first")  


print(f"\nLinhas finais: {len(df)}")
print("\nValores nulos restantes por coluna:")
print(df.isna().sum())

caminho_saida = r"D:\projetos git\analise-de-times-br\data\processed\campeonato-brasileiro-limpo.csv"
df.to_csv(caminho_saida, index=False)

display(df)



C:\Users\Matheus\AppData\Local\Temp\ipykernel_33892\3593738164.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df.select_dtypes(include="object").columns


,vencedor,visitante_Estado,visitante_Placar,hora,formacao_mandante,ID,mandante,tecnico_mandante,mandante_Estado,arrecadacao,visitante,formacao_visitante,data,mandante_Placar,arena,tecnico_visitante,rodata
0,Empate,RJ,1,16:00,4-2-3-1,5839,Sao Paulo,R. Ceni,SP,NaN,Fluminense,4-2-3-1,2017-06-25,1,Morumbi,A. Da Silva Braga,10
1,Empate,MG,1,19:30,NaN,3059,Athletico-Pr,NaN,PR,NaN,Cruzeiro,NaN,2009-11-21,1,Arena Da Baixada,NaN,36
2,Palmeiras,SP,1,19:30,4-2-3-1,8748,Atletico-Go,A. Gomes,GO,NaN,Palmeiras,NaN,2024-11-23,0,Estádio Antônio Accioly - Atlético Goianiense,A. Moreira Ferreira,35
3,Internacional,MG,0,19:10,NaN,2665,Internacional,NaN,RS,NaN,Ipatinga,NaN,2008-11-09,4,Beira Rio,NaN,34
4,Cruzeiro,SP,0,18:00,NaN,47,Cruzeiro,NaN,MG,NaN,Ponte Preta,NaN,2003-04-13,3,Mineirão,NaN,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9984,Figueirense,RJ,0,19:30,3-4-2-1,5734,Figueirense,M. Dos Santos Gonçalves,SC,NaN,Fluminense,4-2-3-1,2016-11-27,1,Orlando Scarpelli,M. De Oliveira,37
9985,Atletico-Mg,MG,2,16:00,4-2-3-1,5192,Fluminense,E. Alves Moreira,RJ,NaN,Atletico-Mg,4-2-3-1,2015-08-30,1,Maracanã,L. Culpi,21
9986,Santa Cruz,MG,1,21:45,4-2-3-1,5391,Santa Cruz,M. Mendes,PE,NaN,Cruzeiro,4-1-2-1-2,2016-05-25,4,Estádio Do Arruda,P. Gomes Bento,3
9987,Goias,SP,0,16:00,NaN,861,Goias,NaN,GO,NaN,Ponte Preta,NaN,2004-08-22,5,Serra Dourada,NaN,26



Linhas finais: 9165

Valores nulos restantes por coluna:
vencedor                 0
visitante_Estado         0
visitante_Placar         0
hora                     0
formacao_mandante     5140
ID                       0
mandante                 0
tecnico_mandante      4821
mandante_Estado          0
arrecadacao           8715
visitante                0
formacao_visitante    5026
data                     0
mandante_Placar         97
arena                    0
tecnico_visitante     4770
rodata                   0
dtype: int64


,vencedor,visitante_Estado,visitante_Placar,hora,formacao_mandante,ID,mandante,tecnico_mandante,mandante_Estado,arrecadacao,visitante,formacao_visitante,data,mandante_Placar,arena,tecnico_visitante,rodata
0,Empate,RJ,1.0,16:00,4-2-3-1,5839,Sao Paulo,R. Ceni,SP,NaN,Fluminense,4-2-3-1,2017-06-25,1.0,Morumbi,A. Da Silva Braga,10
1,Empate,MG,1.0,19:30,NaN,3059,Athletico-Pr,NaN,PR,NaN,Cruzeiro,NaN,2009-11-21,1.0,Arena Da Baixada,NaN,36
2,Palmeiras,SP,1.0,19:30,4-2-3-1,8748,Atletico-Go,A. Gomes,GO,NaN,Palmeiras,NaN,2024-11-23,0.0,Estádio Antônio Accioly - Atlético Goianiense,A. Moreira Ferreira,35
3,Internacional,MG,0.0,19:10,NaN,2665,Internacional,NaN,RS,NaN,Ipatinga,NaN,2008-11-09,4.0,Beira Rio,NaN,34
4,Cruzeiro,SP,0.0,18:00,NaN,47,Cruzeiro,NaN,MG,NaN,Ponte Preta,NaN,2003-04-13,3.0,Mineirão,NaN,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9984,Figueirense,RJ,0.0,19:30,3-4-2-1,5734,Figueirense,M. Dos Santos Gonçalves,SC,NaN,Fluminense,4-2-3-1,2016-11-27,1.0,Orlando Scarpelli,M. De Oliveira,37
9985,Atletico-Mg,MG,2.0,16:00,4-2-3-1,5192,Fluminense,E. Alves Moreira,RJ,NaN,Atletico-Mg,4-2-3-1,2015-08-30,1.0,Maracanã,L. Culpi,21
9986,Santa Cruz,MG,1.0,21:45,4-2-3-1,5391,Santa Cruz,M. Mendes,PE,NaN,Cruzeiro,4-1-2-1-2,2016-05-25,4.0,Estádio Do Arruda,P. Gomes Bento,3
9987,Goias,SP,0.0,16:00,NaN,861,Goias,NaN,GO,NaN,Ponte Preta,NaN,2004-08-22,5.0,Serra Dourada,NaN,26
